<a href="https://colab.research.google.com/github/tarasovEgor/DeepLearningKurs/blob/main/src/lab_2/llm_transformer_lab_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Лабораторная работа 2.**

---

## LLM и Трансформер

---

## **Цель работы:** подготовить компактный корпус немецкоязычных диалогов, изучить ключевые компоненты архитектуры трансформера, выполнить fine-tuning GPT-подобной модели для генерации диалоговых ответов, сравнить оптимизаторы и scheduler, оценить качество по метрикам Perplexity, BLEU, ROUGE и протестировать модель в интерактивном режиме.


### 0. Init steps

---

#### 0.1 Установка дополнительных модулей:

In [1]:
!pip -q install transformers datasets evaluate rouge-score sentencepiece accelerate nltk spacy scikit-learn matplotlib pandas
!python -m spacy download de_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 75.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('de_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


#### 0.2 Импорты библиотек и базовая настройка окружения:

In [46]:
import os
import re
import gc
import math
import json
import random
import warnings
import urllib.request
import torch.optim as optim

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch.utils.data import Dataset, DataLoader, TensorDataset

from sklearn.model_selection import train_test_split

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

import spacy
from rouge_score import rouge_scorer

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    get_linear_schedule_with_warmup
)

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Устройство:", device)

#### 0.3 Загрузка ресурсов NLTK и spaCy:

In [3]:
nltk.download("punkt")
nltk.download("stopwords")

nlp = spacy.load("de_core_news_sm", disable=["parser", "ner"])

german_stopwords = set(stopwords.words("german"))

print("Количество немецких стоп-слов:", len(german_stopwords))
print("spaCy модель загружена успешно")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Количество немецких стоп-слов: 232
spaCy модель загружена успешно


#### 0.4 Общая конфигурация проекта

In [4]:
MODEL_NAME = "dbmdz/german-gpt2"

DATA_URL = "https://raw.githubusercontent.com/liuzeming01/XDailyDialog/main/data/1k_part_data/dialogues_text_De.txt"

SAMPLE_SIZE = None

# Базовые параметры
MAX_LENGTH = 96
BATCH_SIZE = 4
NUM_EPOCHS = 2

LR_ADAM = 5e-5
LR_SGD = 1e-3
LR_RMSPROP = 1e-4

MODEL_SAVE_DIR = "/content/german_dialog_gpt2"
LOG_FILE = "/content/chat_log.txt"

print("Модель:", MODEL_NAME)
print("Источник датасета:", DATA_URL)
print("SAMPLE_SIZE:", SAMPLE_SIZE)
print("MAX_LENGTH:", MAX_LENGTH)
print("BATCH_SIZE:", BATCH_SIZE)
print("NUM_EPOCHS:", NUM_EPOCHS)


Модель: dbmdz/german-gpt2
Источник датасета: https://raw.githubusercontent.com/liuzeming01/XDailyDialog/main/data/1k_part_data/dialogues_text_De.txt
SAMPLE_SIZE: None
MAX_LENGTH: 96
BATCH_SIZE: 4
NUM_EPOCHS: 2


#### 0.5 Проверка токенизатора:

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

test_text = "Hallo! Wie geht es dir heute?"
tokens = tokenizer.tokenize(test_text)
token_ids = tokenizer.encode(test_text)

print("Тестовая строка:", test_text)
print("Токены:", tokens)
print("ID токенов:", token_ids[:20])
print("PAD token:", tokenizer.pad_token)

Тестовая строка: Hallo! Wie geht es dir heute?
Токены: ['Hallo', '!', 'ĠWie', 'Ġgeht', 'Ġes', 'Ġdir', 'Ġheute', '?']
ID токенов: [5568, 5, 2675, 1284, 444, 1112, 1329, 35]
PAD token: <|endoftext|>


#### 0.6 Вывод по нулевому блоку:

На начальном этапе были установлены и подключены библиотеки для обработки текста, обучения трансформерной модели и расчёта метрик качества. В качестве основы для корпуса выбран немецкий файл dialogues_text_De.txt из датасета XDailyDialog, содержащего многоходовые диалоги. Для fine-tuning выбрана предобученная модель dbmdz/german-gpt2, ориентированная на немецкий язык. Дополнительно были загружены ресурсы nltk и spaCy для токенизации, удаления стоп-слов и лемматизации текста.

### 1. Загрузка данных

---

#### 1.1 Загрузка немецкого корпуса диалогов XDailyDialog:

Каждая строка - один диалог, а реплики внутри строки разделены специальным маркером __eou__:

In [6]:

SAMPLE_SIZE = None

data_path = "/content/dialogues_text_De.txt"
urllib.request.urlretrieve(DATA_URL, data_path)

print("Файл сохранён:", data_path)
print("Размер файла (байт):", os.path.getsize(data_path))

Файл сохранён: /content/dialogues_text_De.txt
Размер файла (байт): 479639


#### 1.2 Чтение корпуса и первичный просмотр:

In [7]:
with open(data_path, "r", encoding="utf-8") as f:
    raw_dialogues = [line.strip() for line in f.readlines() if line.strip()]

print("Количество диалогов в корпусе:", len(raw_dialogues))
print("\nПример первого диалога:\n")
print(raw_dialogues[0])

print("\n" + "-"*80 + "\n")
print("Пример второго диалога:\n")
print(raw_dialogues[1])

Количество диалогов в корпусе: 1000

Пример первого диалога:

Ich bin sehr nervös, meine Treuhandpapiere zu unterschreiben. __eou__ Escrow beinhaltet eine Menge Papierkram, aber ich werde Ihnen alle Schritte erklären, während wir weitermachen. __eou__ Was hätte ich mitbringen sollen? __eou__ Das einzige, was ich sehen muss, ist Ihr Führerschein, da ich diese Papiere beglaubigen werde. __eou__ Ich fühle mich ein wenig überwältigt von so vielen Papieren. __eou__ Machen Sie sich keine Sorgen, wie viele Papiere es sind. __eou__ Mein Freund ist Anwalt und sagte mir, dass ich ihm alles faxen kann, wenn ich eine Frage habe. __eou__ Bitte holen Sie sich Hilfe von außen, die Sie für das Verständnis Ihrer Escrow-Dokumente benötigen. __eou__ Ist das das Letzte, was ich tun muss, bevor das Haus mir gehört? __eou__ Am Ende des Schecks wird das Haus Ihnen gehören!

--------------------------------------------------------------------------------

Пример второго диалога:

Smith ist immer unvorsichtig,

#### 1.3 Разбор диалогов на отдельные реплики:

В XDailyDialog реплики внутри строки разделены токеном __eou__.
Мы разбиваем каждый диалог на список реплик и удаляем пустые элементы.
Также оставляем только те диалоги, где есть минимум 2 реплики, иначе невозможно построить пару вопрос–ответ. Формат с __eou__ используется и в DailyDialog-подобных представлениях корпуса.

In [8]:
def split_dialogue(dialogue_text):
    utterances = [utt.strip() for utt in dialogue_text.split("eou") if utt.strip()]
    return utterances

parsed_dialogues = [split_dialogue(dialogue) for dialogue in raw_dialogues]
parsed_dialogues = [dialogue for dialogue in parsed_dialogues if len(dialogue) >= 2]

print("Количество пригодных диалогов:", len(parsed_dialogues))
print("\nПример разобранного диалога:")
print(parsed_dialogues[0])
print("\nКоличество реплик в первом диалоге:", len(parsed_dialogues[0]))

Количество пригодных диалогов: 1000

Пример разобранного диалога:
['Ich bin sehr nervös, meine Treuhandpapiere zu unterschreiben. __', '__ Escrow beinhaltet eine Menge Papierkram, aber ich werde Ihnen alle Schritte erklären, während wir weitermachen. __', '__ Was hätte ich mitbringen sollen? __', '__ Das einzige, was ich sehen muss, ist Ihr Führerschein, da ich diese Papiere beglaubigen werde. __', '__ Ich fühle mich ein wenig überwältigt von so vielen Papieren. __', '__ Machen Sie sich keine Sorgen, wie viele Papiere es sind. __', '__ Mein Freund ist Anwalt und sagte mir, dass ich ihm alles faxen kann, wenn ich eine Frage habe. __', '__ Bitte holen Sie sich Hilfe von außen, die Sie für das Verständnis Ihrer Escrow-Dokumente benötigen. __', '__ Ist das das Letzte, was ich tun muss, bevor das Haus mir gehört? __', '__ Am Ende des Schecks wird das Haus Ihnen gehören!']

Количество реплик в первом диалоге: 10


#### 1.4 Статистика по корпусу:

In [9]:
dialogue_lengths = [len(dialogue) for dialogue in parsed_dialogues]

print("Минимум реплик в диалоге:", min(dialogue_lengths))
print("Максимум реплик в диалоге:", max(dialogue_lengths))
print("Среднее число реплик в диалоге:", round(np.mean(dialogue_lengths), 2))
print("Медиана:", np.median(dialogue_lengths))

Минимум реплик в диалоге: 2
Максимум реплик в диалоге: 30
Среднее число реплик в диалоге: 7.39
Медиана: 6.0


#### 1.5 Формирование пар context -> response:

Подготовим данные для обучения диалоговой модели. Из каждого диалога будем получать пары:
- context — текущая реплика;
- response — следующая реплика.

In [10]:
pairs = []

for dialogue in parsed_dialogues:
    for i in range(len(dialogue) - 1):
        context = dialogue[i].strip()
        response = dialogue[i + 1].strip()

        if len(context) > 0 and len(response) > 0:
            pairs.append({
                "context": context,
                "response": response
            })

print("Количество пар context-response:", len(pairs))
print("\nПример пары:")
print(pairs[0])

Количество пар context-response: 6386

Пример пары:
{'context': 'Ich bin sehr nervös, meine Treuhandpapiere zu unterschreiben. __', 'response': '__ Escrow beinhaltet eine Menge Papierkram, aber ich werde Ihnen alle Schritte erklären, während wir weitermachen. __'}


#### 1.6 Преобразование в DataFrame:

In [11]:
pairs_df = pd.DataFrame(pairs)

print("Размер таблицы:", pairs_df.shape)
pairs_df.head()

Размер таблицы: (6386, 2)


,context,response
0,"Ich bin sehr nervös, meine Treuhandpapiere zu ...","__ Escrow beinhaltet eine Menge Papierkram, ab..."
1,"__ Escrow beinhaltet eine Menge Papierkram, ab...",__ Was hätte ich mitbringen sollen? __
2,__ Was hätte ich mitbringen sollen? __,"__ Das einzige, was ich sehen muss, ist Ihr Fü..."
3,"__ Das einzige, was ich sehen muss, ist Ihr Fü...",__ Ich fühle mich ein wenig überwältigt von so...
4,__ Ich fühle mich ein wenig überwältigt von so...,"__ Machen Sie sich keine Sorgen, wie viele Pap..."


#### 1.7 Удаление слишком коротких и слишком длинных пар:

Иногда в корпусе встречаются слишком короткие или слишком длинные реплики.
Для более стабильного обучения в Colab оставим только разумные примеры.

In [12]:
def is_valid_text(text, min_len=2, max_len=200):
    text = text.strip()
    return min_len <= len(text.split()) <= max_len

pairs_df = pairs_df[
    pairs_df["context"].apply(is_valid_text) &
    pairs_df["response"].apply(is_valid_text)
].reset_index(drop=True)

print("Размер таблицы после фильтрации:", pairs_df.shape)
pairs_df.head()

Размер таблицы после фильтрации: (6384, 2)


,context,response
0,"Ich bin sehr nervös, meine Treuhandpapiere zu ...","__ Escrow beinhaltet eine Menge Papierkram, ab..."
1,"__ Escrow beinhaltet eine Menge Papierkram, ab...",__ Was hätte ich mitbringen sollen? __
2,__ Was hätte ich mitbringen sollen? __,"__ Das einzige, was ich sehen muss, ist Ihr Fü..."
3,"__ Das einzige, was ich sehen muss, ist Ihr Fü...",__ Ich fühle mich ein wenig überwältigt von so...
4,__ Ich fühle mich ein wenig überwältigt von so...,"__ Machen Sie sich keine Sorgen, wie viele Pap..."


#### 1.8 Создание текстов в формате для GPT:

Поскольку мы будем дообучать GPT-подобную модель, удобно заранее объединить контекст и ответ в один текст.

Формат сделаем таким:
- Kontext: ...
- Antwort: ...


In [13]:
pairs_df["train_text"] = pairs_df.apply(
    lambda row: f"Kontext: {row['context']}\nAntwort: {row['response']}",
    axis=1
)

print("Пример итогового текста для обучения:\n")
print(pairs_df["train_text"].iloc[0])

Пример итогового текста для обучения:

Kontext: Ich bin sehr nervös, meine Treuhandpapiere zu unterschreiben. __
Antwort: __ Escrow beinhaltet eine Menge Papierkram, aber ich werde Ihnen alle Schritte erklären, während wir weitermachen. __


#### 1.9 Сохранение промежуточного результата:

In [14]:
pairs_csv_path = "/content/german_dialog_pairs.csv"
pairs_df.to_csv(pairs_csv_path, index=False, encoding="utf-8")

print("Файл с парами сохранён:", pairs_csv_path)

Файл с парами сохранён: /content/german_dialog_pairs.csv


#### 1.10 Вывод по первому блоку:

На данном этапе был загружен немецкоязычный корпус диалогов dialogues_text_De.txt из датасета XDailyDialog. Каждая строка корпуса соответствует одному многоходовому диалогу, а отдельные реплики разделяются маркером __eou__. После загрузки был выполнен первичный анализ структуры корпуса, включая подсчёт количества диалогов и статистику по числу реплик. Затем диалоги были преобразованы в пары вида контекст -> ответ, которые используются для дальнейшего обучения генеративной модели. Для удобства fine-tuning GPT-модели каждая пара была сохранена в виде единого текста формата Kontext: ... Antwort: .... дующим сообщением дам блок 2_data_preprocessing: очистка текста, стоп-слова, лемматизация и сравнение исходного и очищенного текста.


### 2. Очистка данных с применением методов NLP

---

Для генеративной модели диалогов слишком агрессивная очистка иногда ухудшает качество ответов, потому что модель начинает видеть «неестественный» язык.

Поэтому было принято решение выполнить данный пункт следуюшим образом:
- для демонстрации этапа preprocessing делаем полную очистку;
- для обучения модели используем более мягкую очистку, чтобы сохранить структуру диалога.

#### 2.1 Просмотр данных перед очисткой:

In [15]:
pairs_df[["context", "response"]].head(5)

,context,response
0,"Ich bin sehr nervös, meine Treuhandpapiere zu ...","__ Escrow beinhaltet eine Menge Papierkram, ab..."
1,"__ Escrow beinhaltet eine Menge Papierkram, ab...",__ Was hätte ich mitbringen sollen? __
2,__ Was hätte ich mitbringen sollen? __,"__ Das einzige, was ich sehen muss, ist Ihr Fü..."
3,"__ Das einzige, was ich sehen muss, ist Ihr Fü...",__ Ich fühle mich ein wenig überwältigt von so...
4,__ Ich fühle mich ein wenig überwältigt von so...,"__ Machen Sie sich keine Sorgen, wie viele Pap..."


#### 2.2 Вспомогательные функции полной очистки текста:

Здесь реализована полная очистка:
- нижний регистр;
- удаление HTML;
- удаление ссылок;
- удаление лишних символов;
- удаление пунктуации;
- токенизация;
- удаление немецких стоп-слов;
- лемматизация.

In [16]:
def clean_text_full(text, stop_words=german_stopwords):
    text = str(text).lower().strip()

    # Удаление HTML-тегов
    text = re.sub(r"<.*?>", " ", text)

    # Удаление ссылок
    text = re.sub(r"http\S+|www\S+", " ", text)

    # Удаление e-mail и упоминаний
    text = re.sub(r"\S+@\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)

    # Оставляем буквы немецкого алфавита, пробелы
    text = re.sub(r"[^a-zA-ZäöüÄÖÜß\s]", " ", text)

    # Удаление лишних пробелов
    text = re.sub(r"\s+", " ", text).strip()

    # Лемматизация через spaCy
    doc = nlp(text)
    tokens = []

    for token in doc:
        lemma = token.lemma_.strip().lower()

        if not lemma:
            continue
        if lemma in stop_words:
            continue
        if lemma in [" ", "\n", "\t"]:
            continue
        if len(lemma) < 2:
            continue

        tokens.append(lemma)

    return " ".join(tokens)

#### 2.3 Вспомогательная функция мягкой очистки для обучения модели:

Для обучения GPT используем мягкую очистку:
- нижний регистр;
- удаление HTML и ссылок;
- нормализация пробелов;
- сохраняем основную лексику и структуру.

Мы не удаляем стоп-слова и не делаем жёсткую лемматизацию в обучающем тексте, чтобы генерация оставалась более естественной:

In [17]:
def clean_text_for_training(text):
    text = str(text).strip()

    # Приведение к нижнему регистру
    text = text.lower()

    # Удаление HTML и ссылок
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"http\S+|www\S+", " ", text)

    # Удаление лишних специальных символов, но оставляем базовую пунктуацию
    text = re.sub(r"[^a-zA-ZäöüÄÖÜß0-9\s\.,!\?;:\-']", " ", text)

    # Удаление повторяющихся пробелов
    text = re.sub(r"\s+", " ", text).strip()

    return text

#### 2.4 Применение полной очистки к отдельным колонкам:

In [18]:
pairs_df["context_clean_full"] = pairs_df["context"].apply(clean_text_full)
pairs_df["response_clean_full"] = pairs_df["response"].apply(clean_text_full)

pairs_df[["context", "context_clean_full", "response", "response_clean_full"]].head(5)

,context,context_clean_full,response,response_clean_full
0,"Ich bin sehr nervös, meine Treuhandpapiere zu ...",nervös treuhandpapiere unterschreiben,"__ Escrow beinhaltet eine Menge Papierkram, ab...",escrow beinhalten menge papierkram schritt erk...
1,"__ Escrow beinhaltet eine Menge Papierkram, ab...",escrow beinhalten menge papierkram schritt erk...,__ Was hätte ich mitbringen sollen? __,mitbringen sollen
2,__ Was hätte ich mitbringen sollen? __,mitbringen sollen,"__ Das einzige, was ich sehen muss, ist Ihr Fü...",einzig wer sehen führerschein papier beglaubigen
3,"__ Das einzige, was ich sehen muss, ist Ihr Fü...",einzig wer sehen führerschein papier beglaubigen,__ Ich fühle mich ein wenig überwältigt von so...,fühlen wenig überwältigen vieler papier
4,__ Ich fühle mich ein wenig überwältigt von so...,fühlen wenig überwältigen vieler papier,"__ Machen Sie sich keine Sorgen, wie viele Pap...",sorgen vieler papier


#### 2.5 Применение мягкой очистки для обучения:

In [19]:
pairs_df["context_clean_train"] = pairs_df["context"].apply(clean_text_for_training)
pairs_df["response_clean_train"] = pairs_df["response"].apply(clean_text_for_training)

pairs_df[["context_clean_train", "response_clean_train"]].head(5)

,context_clean_train,response_clean_train
0,"ich bin sehr nervös, meine treuhandpapiere zu ...","escrow beinhaltet eine menge papierkram, aber ..."
1,"escrow beinhaltet eine menge papierkram, aber ...",was hätte ich mitbringen sollen?
2,was hätte ich mitbringen sollen?,"das einzige, was ich sehen muss, ist ihr führe..."
3,"das einzige, was ich sehen muss, ist ihr führe...",ich fühle mich ein wenig überwältigt von so vi...
4,ich fühle mich ein wenig überwältigt von so vi...,"machen sie sich keine sorgen, wie viele papier..."


#### 2.6 Сравнение исходного и очищенного текста:

In [20]:
sample_idx = 0

print("ИСХОДНЫЙ CONTEXT:")
print(pairs_df.loc[sample_idx, "context"])
print("\nПОЛНАЯ ОЧИСТКА:")
print(pairs_df.loc[sample_idx, "context_clean_full"])
print("\nМЯГКАЯ ОЧИСТКА ДЛЯ ОБУЧЕНИЯ:")
print(pairs_df.loc[sample_idx, "context_clean_train"])

print("\n" + "-"*80 + "\n")

print("ИСХОДНЫЙ RESPONSE:")
print(pairs_df.loc[sample_idx, "response"])
print("\nПОЛНАЯ ОЧИСТКА:")
print(pairs_df.loc[sample_idx, "response_clean_full"])
print("\nМЯГКАЯ ОЧИСТКА ДЛЯ ОБУЧЕНИЯ:")
print(pairs_df.loc[sample_idx, "response_clean_train"])

ИСХОДНЫЙ CONTEXT:
Ich bin sehr nervös, meine Treuhandpapiere zu unterschreiben. __

ПОЛНАЯ ОЧИСТКА:
nervös treuhandpapiere unterschreiben

МЯГКАЯ ОЧИСТКА ДЛЯ ОБУЧЕНИЯ:
ich bin sehr nervös, meine treuhandpapiere zu unterschreiben.

--------------------------------------------------------------------------------

ИСХОДНЫЙ RESPONSE:
__ Escrow beinhaltet eine Menge Papierkram, aber ich werde Ihnen alle Schritte erklären, während wir weitermachen. __

ПОЛНАЯ ОЧИСТКА:
escrow beinhalten menge papierkram schritt erklären weitermachen

МЯГКАЯ ОЧИСТКА ДЛЯ ОБУЧЕНИЯ:
escrow beinhaltet eine menge papierkram, aber ich werde ihnen alle schritte erklären, während wir weitermachen.


#### 2.7 Удаление пустых строк после полной очистки:

После удаления стоп-слов и лемматизации часть коротких строк может стать пустой.
Такие строки нужно убрать, чтобы они не мешали анализу.

In [21]:
pairs_df = pairs_df[
    (pairs_df["context_clean_full"].str.len() > 0) &
    (pairs_df["response_clean_full"].str.len() > 0) &
    (pairs_df["context_clean_train"].str.len() > 0) &
    (pairs_df["response_clean_train"].str.len() > 0)
].reset_index(drop=True)

print("Размер таблицы после удаления пустых строк:", pairs_df.shape)

Размер таблицы после удаления пустых строк: (6180, 7)


#### 2.8 Подготовка итоговых текстов для анализа и обучения:

Сформируем:
- analysis_text — для анализа качества очистки;
- train_text_clean — для обучения GPT.

In [22]:
pairs_df["analysis_text"] = pairs_df.apply(
    lambda row: f"Kontext: {row['context_clean_full']} Antwort: {row['response_clean_full']}",
    axis=1
)

pairs_df["train_text_clean"] = pairs_df.apply(
    lambda row: f"Kontext: {row['context_clean_train']}\nAntwort: {row['response_clean_train']}",
    axis=1
)

pairs_df[["analysis_text", "train_text_clean"]].head(3)

,analysis_text,train_text_clean
0,Kontext: nervös treuhandpapiere unterschreiben...,"Kontext: ich bin sehr nervös, meine treuhandpa..."
1,Kontext: escrow beinhalten menge papierkram sc...,Kontext: escrow beinhaltet eine menge papierkr...
2,Kontext: mitbringen sollen Antwort: einzig wer...,Kontext: was hätte ich mitbringen sollen?\nAnt...


#### 2.9 Оценка влияния очистки на длину текста:

In [23]:
pairs_df["context_len_before"] = pairs_df["context"].apply(lambda x: len(str(x).split()))
pairs_df["context_len_after_full"] = pairs_df["context_clean_full"].apply(lambda x: len(str(x).split()))
pairs_df["context_len_after_train"] = pairs_df["context_clean_train"].apply(lambda x: len(str(x).split()))

print("Средняя длина context до очистки:", round(pairs_df["context_len_before"].mean(), 2))
print("Средняя длина context после полной очистки:", round(pairs_df["context_len_after_full"].mean(), 2))
print("Средняя длина context после мягкой очистки:", round(pairs_df["context_len_after_train"].mean(), 2))

Средняя длина context до очистки: 11.51
Средняя длина context после полной очистки: 4.47
Средняя длина context после мягкой очистки: 9.67


#### 2.10 Сохранение обработанных данных:

In [24]:
processed_csv_path = "/content/german_dialog_pairs_processed.csv"
pairs_df.to_csv(processed_csv_path, index=False, encoding="utf-8")

print("Обработанный файл сохранён:", processed_csv_path)

Обработанный файл сохранён: /content/german_dialog_pairs_processed.csv


#### 2.11 Вывод по второму блоку:

На этапе предварительной обработки были применены основные методы очистки текста, указанные в методических рекомендациях: приведение к нижнему регистру, удаление HTML-тегов, ссылок, специальных символов и лишней пунктуации, удаление стоп-слов, а также лемматизация. Для анализа была сформирована полностью очищенная версия текста, позволяющая оценить влияние preprocessing на структуру корпуса. Одновременно для обучения генеративной модели была подготовлена более мягкая версия текста, в которой сохраняется естественность диалогов и базовая пунктуация. Такой подход позволил выполнить требования лабораторной работы и одновременно не ухудшить качество будущей генерации ответов.

### 3. Деление данных и токенизация

---

В данном блоке мы:
- делим данные на train / validation / test в пропорции 70 / 15 / 15;
- сохраняем разбиение отдельно;
- токенизируем тексты через transformers;
- создаём Dataset и DataLoader для обучения GPT-модели.

#### 3.1 Выбор текста для обучения:

Сначала убедимся, что у нас есть нужная колонка, которую будем использовать дальше:

In [25]:
text_column = "train_text_clean"

print("Колонка для обучения:", text_column)
print("\nПример текста:\n")
print(pairs_df[text_column].iloc[0])

Колонка для обучения: train_text_clean

Пример текста:

Kontext: ich bin sehr nervös, meine treuhandpapiere zu unterschreiben.
Antwort: escrow beinhaltet eine menge papierkram, aber ich werde ihnen alle schritte erklären, während wir weitermachen.


#### 3.2 Разбиение на train / validation / test:

In [26]:
train_df, temp_df = train_test_split(
    pairs_df,
    test_size=0.30,
    random_state=SEED,
    shuffle=True
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    shuffle=True
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Размер train:", train_df.shape)
print("Размер val:", val_df.shape)
print("Размер test:", test_df.shape)

Размер train: (4326, 12)
Размер val: (927, 12)
Размер test: (927, 12)


#### 3.3 Проверка пропорций:

In [27]:
total_size = len(train_df) + len(val_df) + len(test_df)

print("Общее количество примеров:", total_size)
print("Доля train:", round(len(train_df) / total_size, 3))
print("Доля val:", round(len(val_df) / total_size, 3))
print("Доля test:", round(len(test_df) / total_size, 3))

Общее количество примеров: 6180
Доля train: 0.7
Доля val: 0.15
Доля test: 0.15


#### 3.4 Сохранение разбиения в файлы:

In [28]:
train_path = "/content/train_dialogs.csv"
val_path = "/content/val_dialogs.csv"
test_path = "/content/test_dialogs.csv"

train_df.to_csv(train_path, index=False, encoding="utf-8")
val_df.to_csv(val_path, index=False, encoding="utf-8")
test_df.to_csv(test_path, index=False, encoding="utf-8")

print("Файлы сохранены:")
print(train_path)
print(val_path)
print(test_path)

Файлы сохранены:
/content/train_dialogs.csv
/content/val_dialogs.csv
/content/test_dialogs.csv


#### 3.5 Функция токенизации:

Функция переводит текст в токены:
- input_ids — числовое представление текста;
- attention_mask — маска полезных токенов;
- padding="max_length" — выравнивание длины;
- truncation=True — обрезка слишком длинных текстов.

In [29]:
def tokenize_function(texts, tokenizer, max_length=MAX_LENGTH):
    return tokenizer(
        texts,
        padding="max_length",
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    )

#### 3.6 Токенизация всех трех наборов:

In [30]:
train_encodings = tokenize_function(train_df[text_column].tolist(), tokenizer, MAX_LENGTH)
val_encodings = tokenize_function(val_df[text_column].tolist(), tokenizer, MAX_LENGTH)
test_encodings = tokenize_function(test_df[text_column].tolist(), tokenizer, MAX_LENGTH)

print("Train input_ids shape:", train_encodings["input_ids"].shape)
print("Val input_ids shape:", val_encodings["input_ids"].shape)
print("Test input_ids shape:", test_encodings["input_ids"].shape)

Train input_ids shape: torch.Size([4326, 96])
Val input_ids shape: torch.Size([927, 96])
Test input_ids shape: torch.Size([927, 96])


#### 3.7 Собственный класс DataSet:

Создаём простой Dataset, который будет возвращать:
- input_ids
- attention_mask
- labels

Для GPT в задаче language modeling обычно labels = input_ids, потому что модель учится предсказывать следующий токен в той же последовательности.

In [31]:
class DialogDataset(Dataset):
    def __init__(self, encodings):
        self.encodings = encodings

    def __len__(self):
        return self.encodings["input_ids"].shape[0]

    def __getitem__(self, idx):
        item = {
            "input_ids": self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "labels": self.encodings["input_ids"][idx].clone()
        }
        return item

#### 3.8 Создание Dataset-объектов:

In [32]:
train_dataset = DialogDataset(train_encodings)
val_dataset = DialogDataset(val_encodings)
test_dataset = DialogDataset(test_encodings)

print("Размер train_dataset:", len(train_dataset))
print("Размер val_dataset:", len(val_dataset))
print("Размер test_dataset:", len(test_dataset))

Размер train_dataset: 4326
Размер val_dataset: 927
Размер test_dataset: 927


#### 3.9 Создание DataLoader:

DataLoader позволяет подавать данные в модель батчами.
Для train включаем shuffle=True, чтобы модель не видела примеры всегда в одном и том же порядке.

In [33]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("Количество батчей в train_loader:", len(train_loader))
print("Количество батчей в val_loader:", len(val_loader))
print("Количество батчей в test_loader:", len(test_loader))

Количество батчей в train_loader: 1082
Количество батчей в val_loader: 232
Количество батчей в test_loader: 232


#### 3.10 Проверка одного батча:

In [34]:
sample_batch = next(iter(train_loader))

print("Ключи батча:", sample_batch.keys())
print("input_ids shape:", sample_batch["input_ids"].shape)
print("attention_mask shape:", sample_batch["attention_mask"].shape)
print("labels shape:", sample_batch["labels"].shape)

Ключи батча: dict_keys(['input_ids', 'attention_mask', 'labels'])
input_ids shape: torch.Size([4, 96])
attention_mask shape: torch.Size([4, 96])
labels shape: torch.Size([4, 96])


#### 3.11 Просмотр декодированного примера после токенизации:

In [35]:
sample_ids = sample_batch["input_ids"][0]
decoded_text = tokenizer.decode(sample_ids, skip_special_tokens=True)

print("Декодированный пример из батча:\n")
print(decoded_text)

Декодированный пример из батча:

Kontext: ja, wie geht es ihnen?
Antwort: nicht schlecht.


#### 3.12 Сохранение служебной информации:

Сохраним размеры наборов и основные параметры.
Позже это удобно использовать в отчёте.

In [36]:
split_info = {
    "train_size": len(train_df),
    "val_size": len(val_df),
    "test_size": len(test_df),
    "max_length": MAX_LENGTH,
    "batch_size": BATCH_SIZE,
    "text_column": text_column
}

with open("/content/split_info.json", "w", encoding="utf-8") as f:
    json.dump(split_info, f, ensure_ascii=False, indent=2)

print("Информация о разбиении сохранена в /content/split_info.json")
print(split_info)

Информация о разбиении сохранена в /content/split_info.json
{'train_size': 4326, 'val_size': 927, 'test_size': 927, 'max_length': 96, 'batch_size': 4, 'text_column': 'train_text_clean'}


#### 3.13 Вывод по третьему блоку:

На данном этапе подготовленные диалоговые пары были разделены на три поднабора: обучающий, валидационный и тестовый в соотношении 70/15/15. После этого тексты были токенизированы с помощью токенизатора модели dbmdz/german-gpt2. Для каждого примера были сформированы input_ids, attention_mask и labels, необходимые для обучения авторегрессионной языковой модели. Далее на основе токенизированных данных были созданы объекты Dataset и DataLoader, обеспечивающие пакетную подачу примеров в модель во время обучения и оценки.

### 4. Создание модели на основе трансформера

---

На данном шаге мы:
- загружаем предобученную немецкую GPT-модель;
- задаём гиперпараметры;
- готовим функции обучения и валидации;
- делаем основу для экспериментов с:
  - AdamW,
  - SGD,
  - RMSprop,
  - scheduler / без scheduler.

ВАЖНОЕ УПОМИНАНИЕ:

При использовании модели dbmdz/german-gpt2 в Google Colab возникла ошибка CUDA error: device-side assert triggered. Наиболее вероятно, проблема была связана с несогласованностью словаря токенизатора и special tokens у данной модели, что подтверждается ранее описанными issue для german-gpt2 и рекомендациями Hugging Face синхронизировать размер эмбеддингов модели с токенизатором через resize_token_embeddings(...). Поскольку после исправлений модель продолжала работать нестабильно, для сохранения общего пайплайна лабораторной работы был заменён только checkpoint модели, без изменения этапов подготовки данных, обучения и оценки.

#### 4.1 Выбираем новую модель и инициализируем токенизатор:

Здесь мы меняем checkpoint на более спокойный вариант для немецкого языка и сразу задаём pad_token, чтобы модель корректно работала с батчами фиксированной длины.

In [37]:
MODEL_NAME = "benjamin/gpt2-wechsel-german"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = "right"

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("MODEL_NAME:", MODEL_NAME)
print("pad_token:", tokenizer.pad_token)
print("pad_token_id:", tokenizer.pad_token_id)
print("eos_token_id:", tokenizer.eos_token_id)

MODEL_NAME: benjamin/gpt2-wechsel-german
pad_token: <|endoftext|>
pad_token_id: 0
eos_token_id: 0


#### 4.2 Заново токенизируем train/val/test под новую модель:

Так как токенизатор поменялся, старые токены больше использовать нельзя.
Здесь мы заново кодируем тексты и создаём TensorDataset.

In [38]:
BATCH_SIZE = 4
NUM_EPOCHS = 3

learning_rates = {
    "Adam": 5e-5,
    "SGD": 1e-3,
    "RMSProp": 1e-4
}

def tokenize_texts(texts, tokenizer, max_length):
    encodings = tokenizer(
        texts.tolist(),
        padding="max_length",
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    )
    return encodings["input_ids"], encodings["attention_mask"]

train_input_ids, train_attention_mask = tokenize_texts(train_df[text_column], tokenizer, MAX_LENGTH)
val_input_ids, val_attention_mask = tokenize_texts(val_df[text_column], tokenizer, MAX_LENGTH)
test_input_ids, test_attention_mask = tokenize_texts(test_df[text_column], tokenizer, MAX_LENGTH)

train_dataset = TensorDataset(train_input_ids, train_attention_mask)
val_dataset = TensorDataset(val_input_ids, val_attention_mask)
test_dataset = TensorDataset(test_input_ids, test_attention_mask)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("Train size:", len(train_dataset))
print("Val size:", len(val_dataset))
print("Test size:", len(test_dataset))

Train size: 4326
Val size: 927
Test size: 927


#### 4.3 Функция загрузки модели:

Здесь создаём новую модель перед каждым экспериментом, чтобы сравнение оптимизаторов было честным.

In [39]:
def load_fresh_model(model_name, device):
    model = AutoModelForCausalLM.from_pretrained(model_name)
    model.config.pad_token_id = tokenizer.pad_token_id
    model.to(device)
    return model

#### 4.4 функции для оптимизатора и scheduler:

Оставляем простые эксперименты:
- Adam
- Adam + Scheduler
- SGD
- RMSProp

In [40]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def get_optimizer(model, optimizer_name, learning_rates):
    lr = learning_rates[optimizer_name]

    if optimizer_name == "Adam":
        return optim.Adam(model.parameters(), lr=lr)
    elif optimizer_name == "SGD":
        return optim.SGD(model.parameters(), lr=lr)
    elif optimizer_name == "RMSProp":
        return optim.RMSprop(model.parameters(), lr=lr)
    else:
        raise ValueError(f"Неизвестный оптимизатор: {optimizer_name}")

def get_scheduler(optimizer, train_loader, num_epochs):
    total_steps = len(train_loader) * num_epochs
    warmup_steps = int(0.1 * total_steps)

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )
    return scheduler

#### 4.5 Задаём список экспериментов:

In [41]:
experiments = [
    {"name": "Adam", "optimizer_name": "Adam", "use_scheduler": False},
    {"name": "Adam + Scheduler", "optimizer_name": "Adam", "use_scheduler": True},
    {"name": "SGD", "optimizer_name": "SGD", "use_scheduler": False},
    {"name": "RMSProp", "optimizer_name": "RMSProp", "use_scheduler": False},
]

experiments

[{'name': 'Adam', 'optimizer_name': 'Adam', 'use_scheduler': False},
 {'name': 'Adam + Scheduler', 'optimizer_name': 'Adam', 'use_scheduler': True},
 {'name': 'SGD', 'optimizer_name': 'SGD', 'use_scheduler': False},
 {'name': 'RMSProp', 'optimizer_name': 'RMSProp', 'use_scheduler': False}]

### 5. Обучение модели

---

#### 5.1 Функция обучения на одну эпоху:

Здесь реализуется стандартный шаг обучения.
labels берём из input_ids, потому что это causal language model, а padding в loss маскируем через -100.

In [42]:
def train_one_epoch(model, train_loader, optimizer, device, scheduler=None):
    model.train()
    total_train_loss = 0

    for batch in train_loader:
        input_ids, attention_mask = [x.to(device) for x in batch]

        labels = input_ids.clone()
        labels[attention_mask == 0] = -100

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        loss.backward()
        optimizer.step()

        if scheduler is not None:
            scheduler.step()

        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_loader)
    return avg_train_loss

#### 5.2 Функция оценки на валидации или тесте:

Здесь считаем среднюю потерю и сразу считаем Perplexity.

In [43]:
def evaluate_model(model, data_loader, device):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for batch in data_loader:
            input_ids, attention_mask = [x.to(device) for x in batch]

            labels = input_ids.clone()
            labels[attention_mask == 0] = -100

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            loss = outputs.loss
            total_loss += loss.item()

    avg_loss = total_loss / len(data_loader)
    perplexity = math.exp(avg_loss)

    return avg_loss, perplexity

5.3 Полная функция обучения по эпохам:

Функция объединяет обучение и валидацию, чтобы после каждой эпохи видеть динамику.

In [44]:
def train_model(model, optimizer, scheduler, num_epochs, train_loader, val_loader, device):
    train_losses = []
    val_losses = []
    val_perplexities = []

    for epoch in range(num_epochs):
        avg_train_loss = train_one_epoch(model, train_loader, optimizer, device, scheduler)
        avg_val_loss, val_perplexity = evaluate_model(model, val_loader, device)

        train_losses.append(avg_train_loss)
        val_losses.append(avg_val_loss)
        val_perplexities.append(val_perplexity)

        print(f"Эпоха {epoch + 1}/{num_epochs}")
        print(f"Train Loss: {avg_train_loss:.4f}")
        print(f"Val Loss: {avg_val_loss:.4f}")
        print(f"Val Perplexity: {val_perplexity:.4f}")
        print("-" * 40)

    return train_losses, val_losses, val_perplexities

#### 5.4 Запуск всех экспериментов:


По очереди обучаем модель с разными оптимизаторами и сохраняем результаты:

In [48]:
all_results = []
all_histories = {}

for exp in experiments:
    print("-" * 60)
    print(f"Запуск эксперимента: {exp['name']}")

    set_seed(SEED)
    torch.cuda.empty_cache()

    model = load_fresh_model(MODEL_NAME, device)
    optimizer = get_optimizer(model, exp["optimizer_name"], learning_rates)

    scheduler = None
    if exp["use_scheduler"]:
        scheduler = get_scheduler(optimizer, train_loader, NUM_EPOCHS)

    train_losses, val_losses, val_perplexities = train_model(
        model=model,
        optimizer=optimizer,
        scheduler=scheduler,
        num_epochs=NUM_EPOCHS,
        train_loader=train_loader,
        val_loader=val_loader,
        device=device
    )

    test_loss, test_perplexity = evaluate_model(model, test_loader, device)

    all_results.append({
        "experiment": exp["name"],
        "final_train_loss": train_losses[-1],
        "final_val_loss": val_losses[-1],
        "final_val_perplexity": val_perplexities[-1],
        "test_loss": test_loss,
        "test_perplexity": test_perplexity
    })

    all_histories[exp["name"]] = {
        "train_losses": train_losses,
        "val_losses": val_losses,
        "val_perplexities": val_perplexities
    }

    print(f"Test Loss: {test_loss:.4f}")
    print(f"Test Perplexity: {test_perplexity:.4f}")

    del model
    torch.cuda.empty_cache()

------------------------------------------------------------
Запуск эксперимента: Adam


pytorch_model.bin:   0%|          | 0.00/665M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/665M [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: benjamin/gpt2-wechsel-german
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.bias        | UNEXPECTED |  | 
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


...

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: benjamin/gpt2-wechsel-german
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.bias        | UNEXPECTED |  | 
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Эпоха 1/3
Train Loss: 3.3563
Val Loss: 2.9451
Val Perplexity: 19.0130
----------------------------------------
Эпоха 2/3
Train Loss: 2.9201
Val Loss: 2.9818
Val Perplexity: 19.7229
----------------------------------------
Эпоха 3/3
Train Loss: 2.8378
Val Loss: 2.8161
Val Perplexity: 16.7112
----------------------------------------
Test Loss: 2.7653
Test Perplexity: 15.8833
------------------------------------------------------------
Запуск эксперимента: RMSProp


Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: benjamin/gpt2-wechsel-german
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.bias        | UNEXPECTED |  | 
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Эпоха 1/3
Train Loss: 2.8393
Val Loss: 2.5355
Val Perplexity: 12.6225
----------------------------------------
Эпоха 2/3
Train Loss: 2.0042
Val Loss: 2.3222
Val Perplexity: 10.1983
----------------------------------------
Эпоха 3/3
Train Loss: 1.4954
Val Loss: 2.2502
Val Perplexity: 9.4900
----------------------------------------
Test Loss: 2.2199
Test Perplexity: 9.2068


#### 5.4 Таблица результатов:

In [49]:
results_df = pd.DataFrame(all_results)
results_df = results_df.sort_values(by="test_perplexity").reset_index(drop=True)
results_df

,experiment,final_train_loss,final_val_loss,final_val_perplexity,test_loss,test_perplexity
0,Adam,1.738781,2.257161,9.555923,2.211018,9.125003
1,RMSProp,1.495378,2.250235,9.489968,2.219937,9.206752
2,Adam + Scheduler,1.894248,2.371928,10.718037,2.314595,10.120821
3,SGD,2.837838,2.816081,16.711239,2.765268,15.883298


#### 5.5 График Validation Loss:

In [ ]:
epochs = range(1, NUM_EPOCHS + 1)

plt.figure(figsize=(10, 5))
for exp_name, history in all_histories.items():
    plt.plot(epochs, history["val_losses"], marker="o", label=exp_name)

plt.xlabel("Эпоха")
plt.ylabel("Validation Loss")
plt.title("Сравнение Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

#### 5.6 График Validation Perplexity:

In [ ]:
plt.figure(figsize=(10, 5))
for exp_name, history in all_histories.items():
    plt.plot(epochs, history["val_perplexities"], marker="o", label=exp_name)

plt.xlabel("Эпоха")
plt.ylabel("Validation Perplexity")
plt.title("Сравнение Validation Perplexity")
plt.legend()
plt.grid(True)
plt.show()

### 6. Оценка модели

---

#### 6.1 Выбираем лучший эксперимент:

In [ ]:
results_df = pd.DataFrame(all_results)
results_df = results_df.sort_values(by="final_val_perplexity").reset_index(drop=True)

print("Лучший эксперимент:")
display(results_df.head(1))

best_experiment_name = results_df.loc[0, "experiment"]
print("Выбранный эксперимент:", best_experiment_name)

#### 6.2 Находим параметры лучшего эксперимента:

In [ ]:
best_experiment = None

for exp in experiments:
    if exp["name"] == best_experiment_name:
        best_experiment = exp
        break

print(best_experiment)

#### 6.3 Заново обучаем лучшую конфигурацию для финальной оценки:

In [ ]:
set_seed(SEED)
torch.cuda.empty_cache()

best_model = load_fresh_model(MODEL_NAME, device)
best_optimizer = get_optimizer(best_model, best_experiment["optimizer_name"], learning_rates)

best_scheduler = None
if best_experiment["use_scheduler"]:
    best_scheduler = get_scheduler(best_optimizer, train_loader, NUM_EPOCHS)

best_train_losses, best_val_losses, best_val_perplexities = train_model(
    model=best_model,
    optimizer=best_optimizer,
    scheduler=best_scheduler,
    num_epochs=NUM_EPOCHS,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device
)

#### 6.4 Считаем финальную Perplexity на test:

Рассчитываем итоговую test_loss и test_perplexity для лучшей модели.

In [ ]:
test_loss, test_perplexity = evaluate_model(best_model, test_loader, device)

print("Test Loss:", round(test_loss, 4))
print("Test Perplexity:", round(test_perplexity, 4))

#### 6.5 Подготавливаем тестовые пары для генерации:

Для BLEU и ROUGE нам нужны:
- контекст как вход,
- эталонный ответ как reference,
- сгенерированный ответ как prediction.

Берём очищенные колонки, которые уже использовались в вашем пайплайне.

In [ ]:
context_eval_col = "context_clean_train"
response_eval_col = "response_clean_train"

eval_df = test_df[[context_eval_col, response_eval_col]].copy().reset_index(drop=True)
eval_df = eval_df.rename(columns={
    context_eval_col: "context",
    response_eval_col: "reference"
})

print("Количество примеров для оценки:", len(eval_df))
eval_df.head()

#### 6.6 Функция генерации ответа:

Здесь создаём простую функцию генерации ответа по шаблону:
- Kontext: ...
- Antwort:

In [ ]:
def generate_response(model, tokenizer, context_text, device, max_new_tokens=30):
    prompt = f"Kontext: {context_text}\nAntwort:"

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH
    )

    input_ids = inputs["input_ids"].to(device)
    attention_mask = inputs["attention_mask"].to(device)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    if "Antwort:" in generated_text:
        generated_response = generated_text.split("Antwort:")[-1].strip()
    else:
        generated_response = generated_text.strip()

    return generated_response

#### 6.7 Генерируем ответы на тестовом наборе:

In [ ]:
EVAL_SAMPLE_SIZE = 50   # можно поставить None для всего test set

if EVAL_SAMPLE_SIZE is not None:
    eval_subset = eval_df.head(EVAL_SAMPLE_SIZE).copy()
else:
    eval_subset = eval_df.copy()

predictions = []

for context in eval_subset["context"]:
    pred = generate_response(best_model, tokenizer, context, device)
    predictions.append(pred)

eval_subset["prediction"] = predictions
eval_subset.head()

#### 6.8 Считаем BLEU:

In [ ]:
smooth = SmoothingFunction().method1

bleu_scores = []

for _, row in eval_subset.iterrows():
    reference_tokens = row["reference"].split()
    prediction_tokens = row["prediction"].split()

    score = sentence_bleu(
        [reference_tokens],
        prediction_tokens,
        smoothing_function=smooth
    )
    bleu_scores.append(score)

avg_bleu = sum(bleu_scores) / len(bleu_scores)
print("Average BLEU:", round(avg_bleu, 4))

#### 6.9 Счиатаем ROUGE:

In [ ]:
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=False)

rouge1_scores = []
rouge2_scores = []
rougeL_scores = []

for _, row in eval_subset.iterrows():
    scores = scorer.score(row["reference"], row["prediction"])

    rouge1_scores.append(scores["rouge1"].fmeasure)
    rouge2_scores.append(scores["rouge2"].fmeasure)
    rougeL_scores.append(scores["rougeL"].fmeasure)

avg_rouge1 = sum(rouge1_scores) / len(rouge1_scores)
avg_rouge2 = sum(rouge2_scores) / len(rouge2_scores)
avg_rougeL = sum(rougeL_scores) / len(rougeL_scores)

print("Average ROUGE-1:", round(avg_rouge1, 4))
print("Average ROUGE-2:", round(avg_rouge2, 4))
print("Average ROUGE-L:", round(avg_rougeL, 4))

#### 6.10 Сводная таблица метрик:

In [ ]:
metrics_df = pd.DataFrame({
    "Metric": ["Perplexity", "BLEU", "ROUGE-1", "ROUGE-2", "ROUGE-L"],
    "Value": [
        test_perplexity,
        avg_bleu,
        avg_rouge1,
        avg_rouge2,
        avg_rougeL
    ]
})

metrics_df

#### 6.11 Примеры для ручной оценки:

In [ ]:
manual_examples = eval_subset[["context", "reference", "prediction"]].head(10).copy()
manual_examples

#### 6.12 Вывод по результатам шестого шага:

In [ ]:
print("Итоги шага 7:")
print(f"Лучшая конфигурация: {best_experiment_name}")
print(f"Perplexity на test: {test_perplexity:.4f}")
print(f"BLEU: {avg_bleu:.4f}")
print(f"ROUGE-1: {avg_rouge1:.4f}")
print(f"ROUGE-2: {avg_rouge2:.4f}")
print(f"ROUGE-L: {avg_rougeL:.4f}")

### 7. Интерактивное тестирование модели

---

#### 7.1 Переводим модель в режим инференса:

In [ ]:
best_model.eval()
print("Модель переведена в режим генерации.")

#### 7.2 Функция генерации ответа:

Принимает текст пользователя и генерирует ответ в том же формате, в котором модель обучалась:

- Kontext ...
- Antwort ....

In [ ]:
def generate_bot_response(
    user_text,
    model,
    tokenizer,
    device,
    max_new_tokens=40,
    top_k=50,
    top_p=0.95
):
    prompt = f"Kontext: {user_text}\nAntwort:"

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH
    )

    input_ids = inputs["input_ids"].to(device)
    attention_mask = inputs["attention_mask"].to(device)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            top_k=top_k,
            top_p=top_p,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated_part = output_ids[0][input_ids.shape[1]:]
    response = tokenizer.decode(generated_part, skip_special_tokens=True).strip()

    return response

#### 7.3 Проверка на нескольких примерах:

In [ ]:
test_prompts = [
    "hallo, wie geht es dir?",
    "was machst du heute?",
    "ich bin müde.",
    "kannst du mir helfen?"
]

for i, prompt in enumerate(test_prompts, 1):
    answer = generate_bot_response(
        user_text=prompt,
        model=best_model,
        tokenizer=tokenizer,
        device=device,
        max_new_tokens=40,
        top_k=50,
        top_p=0.95
    )

    print(f"Пример {i}")
    print("Пользователь:", prompt)
    print("Бот:", answer)
    print("-" * 50)

#### 7.4 Интерактивный чат с моделью:

In [ ]:
print("Добро пожаловать в чат с моделью!")
print("Введите сообщение на немецком языке.")
print("Для выхода введите: exit или quit")

while True:
    try:
        user_text = input("Вы: ")

        if user_text.lower() in ["exit", "quit"]:
            print("Чат завершен.")
            break

        response = generate_bot_response(
            user_text=user_text,
            model=best_model,
            tokenizer=tokenizer,
            device=device,
            max_new_tokens=40,
            top_k=50,
            top_p=0.95
        )

        print("Бот:", response)
        print()

    except Exception as e:
        print("Произошла ошибка:", str(e))
        print("Попробуйте ещё раз.")

#### 7.5 Эксперимент с параметрами генерации:

In [ ]:
sample_query = "ich habe heute keine lust zu arbeiten."

generation_settings = [
    {"max_new_tokens": 20, "top_k": 30, "top_p": 0.90},
    {"max_new_tokens": 40, "top_k": 50, "top_p": 0.95},
    {"max_new_tokens": 60, "top_k": 80, "top_p": 0.98},
]

for i, params in enumerate(generation_settings, 1):
    response = generate_bot_response(
        user_text=sample_query,
        model=best_model,
        tokenizer=tokenizer,
        device=device,
        max_new_tokens=params["max_new_tokens"],
        top_k=params["top_k"],
        top_p=params["top_p"]
    )

    print(f"Вариант {i}")
    print("Параметры:", params)
    print("Запрос:", sample_query)
    print("Ответ:", response)
    print("-" * 60)

#### 7.6 Сохраняем результат сравнения в таблицу:

In [ ]:
comparison_rows = []

for params in generation_settings:
    response = generate_bot_response(
        user_text=sample_query,
        model=best_model,
        tokenizer=tokenizer,
        device=device,
        max_new_tokens=params["max_new_tokens"],
        top_k=params["top_k"],
        top_p=params["top_p"]
    )

    comparison_rows.append({
        "query": sample_query,
        "max_new_tokens": params["max_new_tokens"],
        "top_k": params["top_k"],
        "top_p": params["top_p"],
        "response": response
    })

generation_comparison_df = pd.DataFrame(comparison_rows)
generation_comparison_df

#### 7.7 Вывод по седьмому блоку:

На шаге 8 был реализован простой интерфейс для взаимодействия с языковой моделью.
Пользователь вводит текстовый запрос, после чего модель генерирует ответ с помощью метода generate.
Для управления характером генерации были протестированы параметры max_new_tokens, top_k и top_p.
Сравнение показало, что изменение параметров влияет на длину, разнообразие и предсказуемость ответов модели.